In [ ]:
import pandas as pd
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

In [2]:
data_path = "/Users/sana/Downloads/STAT390/Data/CAR_-_EP_Flow_Activity_Queue__Agent_Names"

In [3]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))
df_main = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 'Activity Start Timestamp', 
                                'Queue Name', 'Agent Name', 'Termination Reason'])
df_main

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason


In [ ]:
i=0
for f in files:
    i = i + 1
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df.shape)

In [5]:
df_main["Activity Start Timestamp"] = pd.to_datetime(df_main["Activity Start Timestamp"], errors="coerce")
# Creating a new column 'hour' as it will be useful to visualize peak calling hours
df_main["hour"] = df_main["Activity Start Timestamp"].dt.hour
df_main["weekday"] = df_main["Activity Start Timestamp"].dt.day_name()

In [83]:
##Customer Left --> Code in this cell is created by Meera, for consistency reasons as we are both looking for similar things

customer_left = df_main[df_main['Termination Reason'] == 'Customer Left']

customer_left.info()

<class 'pandas.core.frame.DataFrame'>
Index: 79194 entries, 2075550 to 3222286
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Contact Session ID        79194 non-null  object        
 1   EP Name                   77863 non-null  object        
 2   Flow Name                 3101 non-null   object        
 3   Activity Name             0 non-null      object        
 4   Activity Start Timestamp  79194 non-null  datetime64[ns]
 5   Queue Name                30694 non-null  object        
 6   Agent Name                27231 non-null  object        
 7   Termination Reason        79194 non-null  object        
 8   hour                      79194 non-null  int32         
 9   weekday                   79194 non-null  object        
dtypes: datetime64[ns](1), int32(1), object(8)
memory usage: 6.3+ MB


In [69]:
# Get all calls where customer left with the full log of transfers/menus/etc  
# Code in this cell is created by Meera, for consistency reasons as we are both looking for similar things
#Inner join
calls_customer_left = df_main[df_main["Contact Session ID"].isin(customer_left["Contact Session ID"])].copy()

#Give each session ID an ID number in order of appearance
calls_customer_left["Call ID"] = calls_customer_left["Contact Session ID"].map(
    {id_: i+1 for i, id_ in enumerate(calls_customer_left["Contact Session ID"].unique())}
)
#Separate out the time
calls_customer_left["Time"] = calls_customer_left["Activity Start Timestamp"].dt.strftime("%I:%M:%S %p")
calls_customer_left["Time"] = pd.to_datetime(calls_customer_left["Time"], errors="coerce")

#Calculate Time Difference between rows 
calls_customer_left["Time Difference"] = (
    calls_customer_left.groupby("Contact Session ID")["Time"]
    .diff()
    .dt.total_seconds()  
)

calls_customer_left = calls_customer_left[
    [
        'Call ID',
        'Contact Session ID',
        'EP Name',
        'Flow Name',
        'Activity Name',
        'Queue Name',
        'Termination Reason',
        'Activity Start Timestamp',
        'Time Difference',
        'Agent Name'
    ]
]

/var/folders/js/ymz6mb411xbdbhb1g8hh6rvr0000gn/T/ipykernel_91646/96307539.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  calls_customer_left["Time"] = pd.to_datetime(calls_customer_left["Time"], errors="coerce")


In [70]:
calls_customer_left.head(5)

,Call ID,Contact Session ID,EP Name,Flow Name,Activity Name,Queue Name,Termination Reason,Activity Start Timestamp,Time Difference,Agent Name
404773,1,7cb76bad-f177-4fdb-b0fd-681f9b2cceb0,Main Number Telephony EP,NaN,NaN,NaN,NaN,2025-03-14 08:09:46,NaN,NaN
404774,1,7cb76bad-f177-4fdb-b0fd-681f9b2cceb0,NaN,LACMain,NaN,NaN,NaN,2025-03-14 08:09:46,0.0,NaN
404775,1,7cb76bad-f177-4fdb-b0fd-681f9b2cceb0,Main Number Telephony EP,NaN,LanguageSelectionMenu,NaN,NaN,2025-03-14 08:09:46,0.0,NaN
404776,1,7cb76bad-f177-4fdb-b0fd-681f9b2cceb0,Main Number Telephony EP,LACMain,NaN,NaN,NaN,2025-03-14 08:09:46,0.0,NaN
404779,1,7cb76bad-f177-4fdb-b0fd-681f9b2cceb0,Main Number Telephony EP,NaN,MainMenu,NaN,NaN,2025-03-14 08:09:54,8.0,NaN


In [71]:
'''
Over here, we are looking for the specific scenario where the customer 
listens to the queue music/dialogue and then hangs up after waiting.
'''
# Make sure events are ordered chronologically within each call
calls_customer_left_sorted = calls_customer_left.sort_values(
    ['Call ID', 'Activity Start Timestamp']
).copy()

# Create shifted columns to look at the next event in sequence
calls_customer_left_sorted['next_call_id'] = calls_customer_left_sorted['Call ID'].shift(-1)
calls_customer_left_sorted['next_termination_reason'] = calls_customer_left_sorted['Termination Reason'].shift(-1)

# Filter condition: PlayMOH300s activity followed by Customer Left (within same call)
filtered = calls_customer_left_sorted[
    (calls_customer_left_sorted['Activity Name'] == 'PlayMOH300s') &
    (calls_customer_left_sorted['Call ID'] == calls_customer_left_sorted['next_call_id']) &
    (calls_customer_left_sorted['next_termination_reason'] == 'Customer Left')
]

filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 344 entries, 2076474 to 3218134
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Call ID                   344 non-null    int64         
 1   Contact Session ID        344 non-null    object        
 2   EP Name                   344 non-null    object        
 3   Flow Name                 0 non-null      object        
 4   Activity Name             344 non-null    object        
 5   Queue Name                0 non-null      object        
 6   Termination Reason        0 non-null      object        
 7   Activity Start Timestamp  344 non-null    datetime64[ns]
 8   Time Difference           344 non-null    float64       
 9   Agent Name                0 non-null      object        
 10  next_call_id              344 non-null    float64       
 11  next_termination_reason   344 non-null    object        
dtypes: datetime64[ns]

In [72]:
# Selecting random Call ID to inspect from the filtered list
calls_customer_left.loc[
    calls_customer_left['Call ID'] == 37,
    ['Contact Session ID', 'Activity Name', 'Termination Reason', 'Activity Start Timestamp', 'Time Difference']
]

,Contact Session ID,Activity Name,Termination Reason,Activity Start Timestamp,Time Difference
2075909,25390a18-15db-4e03-b5f2-bdc52d856154,NaN,NaN,2025-03-17 07:59:51,NaN
2075910,25390a18-15db-4e03-b5f2-bdc52d856154,NaN,NaN,2025-03-17 07:59:51,0.0
2075911,25390a18-15db-4e03-b5f2-bdc52d856154,LanguageSelectionMenu,NaN,2025-03-17 07:59:51,0.0
2075912,25390a18-15db-4e03-b5f2-bdc52d856154,NaN,NaN,2025-03-17 07:59:51,0.0
2075914,25390a18-15db-4e03-b5f2-bdc52d856154,MainMenu,NaN,2025-03-17 08:00:02,11.0
2075922,25390a18-15db-4e03-b5f2-bdc52d856154,NaN,NaN,2025-03-17 08:00:24,22.0
2075923,25390a18-15db-4e03-b5f2-bdc52d856154,SeniorsMenu,NaN,2025-03-17 08:00:24,0.0
2075925,25390a18-15db-4e03-b5f2-bdc52d856154,SeniorsConfirmationMenu,NaN,2025-03-17 08:00:29,5.0
2075940,25390a18-15db-4e03-b5f2-bdc52d856154,SuburbsOrCityMenu,NaN,2025-03-17 08:00:46,17.0
2075950,25390a18-15db-4e03-b5f2-bdc52d856154,SeniorsADAPTMenu,NaN,2025-03-17 08:01:05,19.0


In [82]:
# finding average total call time for calls matching the filtered criteria

# Step 1: Get matching Call IDs
call_ids = filtered['Call ID'].unique()

# Step 2: Subset the original dataframe
subset = calls_customer_left[calls_customer_left['Call ID'].isin(call_ids)]

# Step 3: Compute total time per call
total_time_per_call_in_queue = subset.groupby('Call ID')['Time Difference'].sum()

# Step 4: Compute the average total time
average_call_time_in_queue = total_time_per_call_in_queue.mean()

print(f"Average total call time for matching calls: {average_call_time_in_queue:.2f}")

Average total call time for matching calls: 970.17


In [76]:
# Filter condition: Closed queue dialogue activity followed by Customer Left (within same call)
filtered2 = calls_customer_left_sorted[
    (calls_customer_left_sorted['Activity Name'] == 'ClosedQueueMenu') &
    (calls_customer_left_sorted['Call ID'] == calls_customer_left_sorted['next_call_id']) &
    (calls_customer_left_sorted['next_termination_reason'] == 'Customer Left')
]

filtered2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12848 entries, 2076701 to 3221679
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Call ID                   12848 non-null  int64         
 1   Contact Session ID        12848 non-null  object        
 2   EP Name                   12848 non-null  object        
 3   Flow Name                 0 non-null      object        
 4   Activity Name             12848 non-null  object        
 5   Queue Name                0 non-null      object        
 6   Termination Reason        0 non-null      object        
 7   Activity Start Timestamp  12848 non-null  datetime64[ns]
 8   Time Difference           12848 non-null  float64       
 9   Agent Name                0 non-null      object        
 10  next_call_id              12848 non-null  float64       
 11  next_termination_reason   12848 non-null  object        
dtypes: datetime64[n

In [ ]:
# inspecting a Call ID from the filtered list of customers leaving after closed queue menu
calls_customer_left.loc[
    calls_customer_left['Call ID'] == 72,
    ['Contact Session ID', 'Activity Name', 'Termination Reason', 'Activity Start Timestamp', 'Time Difference']
]

,Contact Session ID,Activity Name,Termination Reason,Activity Start Timestamp,Time Difference
2076644,7290128c-3ff7-4d1f-a5b0-dc8bd24b1a68,NaN,NaN,2025-03-17 08:33:03,NaN
2076645,7290128c-3ff7-4d1f-a5b0-dc8bd24b1a68,NaN,NaN,2025-03-17 08:33:03,0.0
2076646,7290128c-3ff7-4d1f-a5b0-dc8bd24b1a68,LanguageSelectionMenu,NaN,2025-03-17 08:33:03,0.0
2076647,7290128c-3ff7-4d1f-a5b0-dc8bd24b1a68,NaN,NaN,2025-03-17 08:33:03,0.0
2076648,7290128c-3ff7-4d1f-a5b0-dc8bd24b1a68,MainMenu,NaN,2025-03-17 08:33:15,12.0
2076659,7290128c-3ff7-4d1f-a5b0-dc8bd24b1a68,NaN,NaN,2025-03-17 08:33:39,24.0
2076660,7290128c-3ff7-4d1f-a5b0-dc8bd24b1a68,SeniorsMenu,NaN,2025-03-17 08:33:39,0.0
2076662,7290128c-3ff7-4d1f-a5b0-dc8bd24b1a68,SeniorsConfirmationMenu,NaN,2025-03-17 08:33:45,6.0
2076668,7290128c-3ff7-4d1f-a5b0-dc8bd24b1a68,SuburbsOrCityMenu,NaN,2025-03-17 08:34:01,16.0
2076680,7290128c-3ff7-4d1f-a5b0-dc8bd24b1a68,SeniorsADAPTMenu,NaN,2025-03-17 08:34:20,19.0


In [80]:
# finding average total call time for calls matching the filtered2 criteria

# Step 1: Get matching Call IDs
call_ids = filtered2['Call ID'].unique()

# Step 2: Subset the original dataframe
subset = calls_customer_left[calls_customer_left['Call ID'].isin(call_ids)]

# Step 3: Compute total time per call
total_time_per_call = subset.groupby('Call ID')['Time Difference'].sum()

# Step 4: Compute the average total time
average_call_time = total_time_per_call.mean()

print(f"Average total call time for matching calls: {average_call_time:.2f}")

Average total call time for matching calls: 255.00


In [93]:
calls_customer_left.loc[
    calls_customer_left['Call ID'] == 6543,
    ['Contact Session ID', 'Activity Name', 'Termination Reason', 'Activity Start Timestamp', 'Time Difference', 'Agent Name']
]

,Contact Session ID,Activity Name,Termination Reason,Activity Start Timestamp,Time Difference,Agent Name
2181757,8c95373c-017e-4058-83d9-fbcb83b3ed60,NaN,NaN,2025-04-15 14:15:52,NaN,NaN
2181758,8c95373c-017e-4058-83d9-fbcb83b3ed60,NaN,NaN,2025-04-15 14:15:52,0.0,NaN
2181759,8c95373c-017e-4058-83d9-fbcb83b3ed60,LanguageSelectionMenu,NaN,2025-04-15 14:15:52,0.0,NaN
2181760,8c95373c-017e-4058-83d9-fbcb83b3ed60,NaN,NaN,2025-04-15 14:15:52,0.0,NaN
2181767,8c95373c-017e-4058-83d9-fbcb83b3ed60,MainMenu,NaN,2025-04-15 14:16:05,13.0,NaN
2181780,8c95373c-017e-4058-83d9-fbcb83b3ed60,ClinicVoicemailTransfer,NaN,2025-04-15 14:16:57,52.0,NaN
2181781,8c95373c-017e-4058-83d9-fbcb83b3ed60,NaN,NaN,2025-04-15 14:16:57,0.0,NaN
2181782,8c95373c-017e-4058-83d9-fbcb83b3ed60,NaN,NaN,2025-04-15 14:16:57,0.0,NaN
2181783,8c95373c-017e-4058-83d9-fbcb83b3ed60,NaN,NaN,2025-04-15 14:16:58,1.0,CBT Agent
2181788,8c95373c-017e-4058-83d9-fbcb83b3ed60,NaN,NaN,2025-04-15 14:17:16,18.0,CBT Agent


In [ ]:
# finding calls in which customers left and reached the TenantDeterrenceMenu at some point
tenant_calls = calls_customer_left_sorted.loc[
    calls_customer_left_sorted['Activity Name'] == 'TenantDeterrenceMenu', 
    'Call ID'
].unique()
tenant_subset = calls_customer_left_sorted[
    calls_customer_left_sorted['Call ID'].isin(tenant_calls)
]

In [111]:
# Customers who left immediately after TenantDeterranceMenu
left_immediately = tenant_subset[
    (tenant_subset['Activity Name'] == 'TenantDeterranceMenu') &
    (tenant_subset['Call ID'] == tenant_subset['next_call_id']) &
    (tenant_subset['next_termination_reason'] == 'Customer Left')
]

# Customers who went to TenantMenu first, then left
left_after_menu = tenant_subset[
    (tenant_subset['Activity Name'] == 'TenantMenu') &
    (tenant_subset['Call ID'] == tenant_subset['next_call_id']) &
    (tenant_subset['next_termination_reason'] == 'Customer Left')
]
print(len(left_immediately), "calls left immediately after TenantDeterranceMenu")
print(len(left_after_menu), "calls left after reaching TenantMenu")


0 calls left immediately after TenantDeterranceMenu
444 calls left after reaching TenantMenu


In [ ]:
# inspecting a Call ID from the tenant deterrance menu scenario
calls_customer_left.loc[
    calls_customer_left['Call ID'] == 4444,
    ['Contact Session ID', 'Activity Name', 'Termination Reason', 'Activity Start Timestamp', 'Time Difference', 'Agent Name']
]

,Contact Session ID,Activity Name,Termination Reason,Activity Start Timestamp,Time Difference,Agent Name
2148749,43fea056-10d1-4a96-8d3d-eeaf83cc9906,NaN,NaN,2025-04-02 16:00:10,NaN,NaN
2148750,43fea056-10d1-4a96-8d3d-eeaf83cc9906,NaN,NaN,2025-04-02 16:00:10,0.0,NaN
2148751,43fea056-10d1-4a96-8d3d-eeaf83cc9906,LanguageSelectionMenu,NaN,2025-04-02 16:00:10,0.0,NaN
2148752,43fea056-10d1-4a96-8d3d-eeaf83cc9906,NaN,NaN,2025-04-02 16:00:11,1.0,NaN
2148761,43fea056-10d1-4a96-8d3d-eeaf83cc9906,LanguageSelectionMenu,NaN,2025-04-02 16:00:29,18.0,NaN
2148762,43fea056-10d1-4a96-8d3d-eeaf83cc9906,MainMenu,NaN,2025-04-02 16:00:30,1.0,NaN
2148766,43fea056-10d1-4a96-8d3d-eeaf83cc9906,NaN,NaN,2025-04-02 16:01:00,30.0,NaN
2148767,43fea056-10d1-4a96-8d3d-eeaf83cc9906,SeniorsMenu,NaN,2025-04-02 16:01:00,0.0,NaN
2148770,43fea056-10d1-4a96-8d3d-eeaf83cc9906,SeniorsMenu,NaN,2025-04-02 16:01:10,10.0,NaN
2148771,43fea056-10d1-4a96-8d3d-eeaf83cc9906,NaN,NaN,2025-04-02 16:01:12,2.0,NaN


In [121]:
# finding calls in which customers left after reaching the farmworker main menu
left_after_farmworker = calls_customer_left_sorted[
    (calls_customer_left_sorted['Activity Name'] == 'FarmworkerMainMenu') &
    (calls_customer_left_sorted['Call ID'] == calls_customer_left_sorted['next_call_id']) &
    (calls_customer_left_sorted['next_termination_reason'] == 'Customer Left')
]


In [ ]:
# inspecting a Call ID from the farmworker main menu scenario
calls_customer_left.loc[
    calls_customer_left['Call ID'] == 35345,
    ['Contact Session ID', 'Activity Name', 'Termination Reason', 'Activity Start Timestamp', 'Time Difference', 'Agent Name']
]

,Contact Session ID,Activity Name,Termination Reason,Activity Start Timestamp,Time Difference,Agent Name
2660567,99a6c025-3417-4f0a-a952-c42f4756a712,NaN,NaN,2025-07-01 17:43:57,NaN,NaN
2660568,99a6c025-3417-4f0a-a952-c42f4756a712,NaN,NaN,2025-07-01 17:43:57,0.0,NaN
2660569,99a6c025-3417-4f0a-a952-c42f4756a712,LanguageSelectionMenu,NaN,2025-07-01 17:43:57,0.0,NaN
2660570,99a6c025-3417-4f0a-a952-c42f4756a712,NaN,NaN,2025-07-01 17:43:57,0.0,NaN
2660571,99a6c025-3417-4f0a-a952-c42f4756a712,FarmworkerMainMenu,NaN,2025-07-01 17:44:04,7.0,NaN
2660591,99a6c025-3417-4f0a-a952-c42f4756a712,NaN,Customer Left,2025-07-01 18:12:13,1689.0,NaN


In [115]:
# inspecting a Call ID from the farmworker main menu scenario
calls_customer_left.loc[
    calls_customer_left['Call ID'] == 10,
    ['Contact Session ID', 'Activity Name', 'Termination Reason', 'Activity Start Timestamp', 'Time Difference', 'Agent Name']
]

,Contact Session ID,Activity Name,Termination Reason,Activity Start Timestamp,Time Difference,Agent Name
2075617,b4ded8c4-0dc0-4686-889b-0d9b7e9ab265,NaN,NaN,2025-03-16 09:40:14,NaN,NaN
2075618,b4ded8c4-0dc0-4686-889b-0d9b7e9ab265,NaN,NaN,2025-03-16 09:40:14,0.0,NaN
2075619,b4ded8c4-0dc0-4686-889b-0d9b7e9ab265,LanguageSelectionMenu,NaN,2025-03-16 09:40:14,0.0,NaN
2075620,b4ded8c4-0dc0-4686-889b-0d9b7e9ab265,NaN,NaN,2025-03-16 09:40:14,0.0,NaN
2075621,b4ded8c4-0dc0-4686-889b-0d9b7e9ab265,FarmworkerMainMenu,NaN,2025-03-16 09:40:22,8.0,NaN
2075622,b4ded8c4-0dc0-4686-889b-0d9b7e9ab265,NaN,Customer Left,2025-03-16 10:02:22,1320.0,NaN


In [120]:
# looking at all activities after which customers left overall
# Filter rows where the next termination reason is 'Customer Left'
left_after_activity = calls_customer_left_sorted[
    (calls_customer_left_sorted['next_termination_reason'] == 'Customer Left') &
    (calls_customer_left_sorted['Call ID'] == calls_customer_left_sorted['next_call_id'])
]

# Get unique activity names where this happened
unique_activities_left_after = left_after_activity['Activity Name'].unique()

#print("Unique activities after which customers left:")
#print(unique_activities_left_after)

left_counts = (
    left_after_activity['Activity Name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'Activity Name', 'Activity Name': 'Count'})
)

left_counts.head(20)



,Count,count
0,ClosedQueueMenu,12848
1,DisconnectContact1,4140
2,MainMenu,3758
3,LegalMenu2,3578
4,LanguageSelectionMenu,3445
5,FarmworkerMainMenu,3400
6,LegalServerScreenPop,3104
7,ClosedMenu,2192
8,OtherLegalMenu,2048
9,OtherLegalOtherMenu,923
